# Enrichment Analyses - Optimized Method 2 (Shared-s DirMult)

Ranking metric: `-log10(p_value_chisq)` (conditional LRT, shared-s) with `scoreType = "pos"`.
Not composite score, not bootstrap p-values.

Rationale: bootstrap p-values are clipped at 1/200 = 0.005, creating ties at the top
of the ranking that degrade GSEA resolution. The chi-squared p-values span the full
range (min ~7e-257) and are available for all converged genes.

Databases: GO BP, KEGG, Disease Ontology, DisGeNET, REACTOME, MSigDB Hallmark, HPO.
Both GSEA (fgsea) and ORA (hypergeometric).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
    REPO_ROOT <- normalizePath(getwd())
    while (!file.exists(file.path(REPO_ROOT, "environment", "apa-python.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
        REPO_ROOT <- dirname(REPO_ROOT)
    }
    Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR <- file.path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026/03-optimized_method_2")
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE)
setwd(WORK_DIR)

In [ ]:
# Data manipulation packages
library(tidyverse)
library(reshape2)
library(stringr)
# GSEA and GO libraries
library(clusterProfiler)
library(biomaRt)
library(org.Hs.eg.db)
library(org.Mm.eg.db)
library(DOSE)
library(enrichplot)
library(ReactomePA)
library(msigdbr)
# Plotting libraries
library(ggplot2)
library(viridis)
library(hrbrthemes)
# Overlap visualization
library(UpSetR)


## Load data and prepare ranking


In [ ]:
input_path <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized/dirmult_regression_optimized2_joint_df5.csv")
data_dir <- dirname(input_path)
output_dir_v2 <- file.path(data_dir, "ORA")
dir.create(output_dir_v2, showWarnings = FALSE, recursive = TRUE)
metric_v2 <- "neglog10pval_chisq_optimized2"

mouse_to_human_genes <- readr::read_csv(paste0(DATA_ROOT, "/common_files/annotations/mouse_to_human_genes.txt"))


In [ ]:
dm2 <- readr::read_csv(input_path)
cat("Loaded", nrow(dm2), "genes\n")

# Filter non-converged
dm2 <- dm2 %>% filter(converged == TRUE)
cat("After converged filter:", nrow(dm2), "\n")

# Filter multi-gene entries
dm2 <- dm2 %>% filter(!grepl(",", utr_name))
cat("After comma filter:", nrow(dm2), "\n")

# Recompute padj_chisq from p_value_chisq via BH
dm2$padj_chisq <- p.adjust(dm2$p_value_chisq, method = "BH")

# Clamp p_value_chisq == 0 to smallest non-zero
min_p <- min(dm2$p_value_chisq[dm2$p_value_chisq > 0])
dm2$p_value_chisq[dm2$p_value_chisq == 0] <- min_p
cat("Clamped p=0 to", min_p, "\n")

# GSEA ranking: -log10(p_value_chisq) (the main difference from method 1)
dm2$rank_score <- -log10(dm2$p_value_chisq)
dm2 <- arrange(dm2, desc(rank_score))

# Set padj to padj_chisq for ORA thresholding
dm2$padj <- dm2$padj_chisq

cat("\nDiscovery counts (padj_chisq < 0.05):", sum(dm2$padj_chisq < 0.05), "\n")
cat("Ranking range:", round(min(dm2$rank_score), 2), "to", round(max(dm2$rank_score), 2), "\n")


In [ ]:
# Gene ID mappings
dm2$gene_symbol <- mapIds(org.Mm.eg.db, dm2$utr_name, "SYMBOL", "ENSEMBL")

one2one <- filter(mouse_to_human_genes, grepl("ortholog_one2one|ortholog_one2many", `Human homology type`))
one2one_in_data <- one2one[one2one$`Gene stable ID` %in% dm2$utr_name, ] %>%
        arrange(desc(`Human orthology confidence [0 low, 1 high]`), desc(`Human Whole-genome alignment coverage`)) %>%
        subset(!duplicated(`Gene stable ID`))
human_ids <- one2one_in_data[match(dm2$utr_name, one2one_in_data$`Gene stable ID`), ]
dm2$entrezid <- as.numeric(as.character(mapIds(org.Hs.eg.db, human_ids$`Human gene stable ID`, "ENTREZID", "ENSEMBL")))

# Mouse ENTREZ IDs (for KEGG)
dm2$mouse_entrezid <- mapIds(org.Mm.eg.db, dm2$utr_name, "ENTREZID", "ENSEMBL")

# Build named score vectors
score_go_v2 <- setNames(dm2$rank_score, dm2$gene_symbol)
score_go_v2 <- score_go_v2[!is.na(names(score_go_v2))]
score_go_v2 <- score_go_v2[!duplicated(names(score_go_v2))]

score_human_v2 <- setNames(dm2$rank_score, dm2$entrezid)
score_human_v2 <- score_human_v2[!duplicated(names(score_human_v2))]
score_human_v2 <- score_human_v2[!is.na(names(score_human_v2))]

score_kegg_v2 <- setNames(dm2$rank_score, dm2$mouse_entrezid)
score_kegg_v2 <- score_kegg_v2[!is.na(names(score_kegg_v2))]
score_kegg_v2 <- score_kegg_v2[!duplicated(names(score_kegg_v2))]

cat(
        "Score vectors: GO =", length(score_go_v2), ", Human =", length(score_human_v2),
        ", KEGG =", length(score_kegg_v2), "\n"
)


## GSEA - 7 databases


In [ ]:
# gseGO - Biological Process
gsea_go_v2 <- gseGO(
        geneList = score_go_v2, ont = "BP", OrgDb = org.Mm.eg.db, keyType = "SYMBOL",
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 50, maxGSSize = 500
)
cat("GO BP terms:", nrow(gsea_go_v2@result), "\n")
head(gsea_go_v2@result[1:9])


In [ ]:
# gseDO + gseDGN (Disease Ontology + DisGeNET)
gsea_do_v2 <- gseDO(
        score_human_v2,
        minGSSize = 50, maxGSSize = 500,
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos"
)
cat("DO terms:", nrow(gsea_do_v2@result), "\n")
head(gsea_do_v2@result[1:9])

gsea_dgn_v2 <- gseDGN(
        score_human_v2,
        minGSSize = 50, maxGSSize = 500,
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos"
)
cat("DGN terms:", nrow(gsea_dgn_v2@result), "\n")
head(gsea_dgn_v2@result[1:9])

gsea_do_read_v2 <- setReadable(gsea_do_v2, "org.Hs.eg.db", "ENTREZID")
gsea_dgn_read_v2 <- setReadable(gsea_dgn_v2, "org.Hs.eg.db", "ENTREZID")


In [ ]:
# Search for autism/neurodevelopmental terms across all disease databases
search_terms <- "autis|neurodevelop|intellectual|speech|language|cogniti"

for (db_name in c("DO", "DisGeNET")) {
    res <- if (db_name == "DO") gsea_do_v2@result else gsea_dgn_v2@result
    hits <- res %>% filter(grepl(search_terms, Description, ignore.case = TRUE))
    cat(sprintf("\n%s: %d terms matching '%s'\n", db_name, nrow(hits), search_terms))
    if (nrow(hits) > 0) {
        hits %>%
            dplyr::select(ID, Description, p.adjust, NES) %>%
            print()
    }
}


In [ ]:
# gseKEGG - mouse pathways
gsea_kegg_v2 <- gseKEGG(
        geneList = score_kegg_v2, organism = "mmu",
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 50, maxGSSize = 500
)
gsea_kegg_v2@result$Description <- gsub(
        " - Mus musculus \\(house mouse\\)$", "",
        gsea_kegg_v2@result$Description
)
cat("KEGG pathways:", nrow(gsea_kegg_v2@result), "\n")
head(gsea_kegg_v2@result[1:9])

gsea_kegg_read_v2 <- setReadable(gsea_kegg_v2, "org.Mm.eg.db", "ENTREZID")


In [ ]:
# gsePathway (REACTOME) - human ENTREZ IDs
gsea_reactome_v2 <- gsePathway(
        geneList = score_human_v2, organism = "human",
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 20, maxGSSize = 500
)
cat("REACTOME pathways:", nrow(gsea_reactome_v2@result), "\n")
head(gsea_reactome_v2@result[1:9])

gsea_reactome_read_v2 <- setReadable(gsea_reactome_v2, "org.Hs.eg.db", "ENTREZID")


In [ ]:
# MSigDB Hallmark gene sets (mouse orthologs via msigdbr)
hallmark_sets <- msigdbr(species = "mouse", collection = "H")
hallmark_t2g <- hallmark_sets %>%
        dplyr::select(gs_name, gene_symbol) %>%
        dplyr::distinct()
hallmark_t2n <- hallmark_t2g %>%
        dplyr::select(gs_name) %>%
        dplyr::distinct() %>%
        dplyr::mutate(gs_description = gs_name %>%
                str_remove("^HALLMARK_") %>%
                str_replace_all("_", " ") %>%
                str_to_title())

gsea_hallmark_v2 <- GSEA(
        geneList = score_go_v2,
        TERM2GENE = hallmark_t2g,
        TERM2NAME = hallmark_t2n,
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 20, maxGSSize = 500
)
cat("MSigDB Hallmark terms:", nrow(gsea_hallmark_v2@result), "\n")
head(gsea_hallmark_v2@result[1:9])


In [ ]:
# MSigDB HPO gene sets (Human Phenotype Ontology)
hpo_sets <- msigdbr(species = "mouse", collection = "C5", subcollection = "HPO")
hpo_t2g <- hpo_sets %>%
        dplyr::select(gs_name, gene_symbol) %>%
        dplyr::distinct()
hpo_t2n <- hpo_t2g %>%
        dplyr::select(gs_name) %>%
        dplyr::distinct() %>%
        dplyr::mutate(gs_description = gs_name %>%
                str_remove("^HP_") %>%
                str_replace_all("_", " ") %>%
                str_to_title())

gsea_hpo_v2 <- GSEA(
        geneList = score_go_v2,
        TERM2GENE = hpo_t2g,
        TERM2NAME = hpo_t2n,
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 50, maxGSSize = 500
)
cat("HPO terms:", nrow(gsea_hpo_v2@result), "\n")
head(gsea_hpo_v2@result[1:9])

# Search HPO for neurodevelopmental terms
hpo_hits <- gsea_hpo_v2@result %>%
        filter(grepl("autis|neurodevelop|intellectual|speech|language|cogniti", Description, ignore.case = TRUE))
cat("\nHPO neurodevelopmental hits:", nrow(hpo_hits), "\n")
if (nrow(hpo_hits) > 0) print(hpo_hits %>% dplyr::select(ID, Description, p.adjust, NES))


## Over-Representation Analysis (ORA)


In [ ]:
# ORA - uses padj_chisq < 0.05 as significance threshold
sig_genes <- dm2 %>%
        filter(padj < 0.05) %>%
        pull(gene_symbol) %>%
        na.omit() %>%
        unique()
all_genes <- dm2$gene_symbol[!is.na(dm2$gene_symbol)] %>% unique()
cat("Significant genes:", length(sig_genes), "out of", length(all_genes), "\n")

# ORA GO BP
ora_go_v2 <- enrichGO(
        gene = sig_genes, universe = all_genes,
        OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "BP",
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 50, maxGSSize = 500
)
cat("ORA GO BP terms (padj<0.05):", sum(ora_go_v2@result$p.adjust < 0.05), "\n")

# ORA DO + DGN
sig_human <- dm2 %>%
        filter(padj < 0.05) %>%
        pull(entrezid) %>%
        na.omit() %>%
        unique() %>%
        as.character()
all_human <- dm2$entrezid[!is.na(dm2$entrezid)] %>%
        unique() %>%
        as.character()

ora_do_v2 <- enrichDO(
        gene = sig_human, universe = all_human,
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 50, maxGSSize = 500
)
cat("ORA DO terms (padj<0.05):", sum(ora_do_v2@result$p.adjust < 0.05), "\n")

ora_dgn_v2 <- enrichDGN(
        gene = sig_human, universe = all_human,
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 50, maxGSSize = 500
)
cat("ORA DGN terms (padj<0.05):", sum(ora_dgn_v2@result$p.adjust < 0.05), "\n")

# ORA KEGG
sig_kegg <- dm2 %>%
        filter(padj < 0.05) %>%
        pull(mouse_entrezid) %>%
        na.omit() %>%
        unique() %>%
        as.character()
all_kegg <- dm2$mouse_entrezid[!is.na(dm2$mouse_entrezid)] %>%
        unique() %>%
        as.character()

ora_kegg_v2 <- enrichKEGG(
        gene = sig_kegg, universe = all_kegg, organism = "mmu",
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 50, maxGSSize = 500
)
ora_kegg_v2@result$Description <- gsub(" - Mus musculus \\(house mouse\\)$", "", ora_kegg_v2@result$Description)
cat("ORA KEGG terms (padj<0.05):", sum(ora_kegg_v2@result$p.adjust < 0.05), "\n")

# ORA REACTOME
ora_reactome_v2 <- enrichPathway(
        gene = sig_human, universe = all_human, organism = "human",
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 20, maxGSSize = 500
)
cat("ORA REACTOME terms (padj<0.05):", sum(ora_reactome_v2@result$p.adjust < 0.05), "\n")

# ORA Hallmark
ora_hallmark_v2 <- enricher(
        gene = sig_genes, universe = all_genes,
        TERM2GENE = hallmark_t2g, TERM2NAME = hallmark_t2n,
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 20, maxGSSize = 500
)
cat("ORA Hallmark terms (padj<0.05):", sum(ora_hallmark_v2@result$p.adjust < 0.05), "\n")

# ORA HPO
ora_hpo_v2 <- enricher(
        gene = sig_genes, universe = all_genes,
        TERM2GENE = hpo_t2g, TERM2NAME = hpo_t2n,
        pvalueCutoff = 0.05, pAdjustMethod = "BH", minGSSize = 50, maxGSSize = 500
)
cat("ORA HPO terms (padj<0.05):", sum(ora_hpo_v2@result$p.adjust < 0.05), "\n")


## Sensitivity check: high-UMI genes


In [ ]:
dm2_hc <- dm2 %>% filter(UMIs >= 100)
cat("Genes after UMI >= 100 filter:", nrow(dm2_hc), "(dropped", nrow(dm2) - nrow(dm2_hc), ")\n")

score_go_hc <- setNames(dm2_hc$rank_score, dm2_hc$gene_symbol)
score_go_hc <- score_go_hc[!is.na(names(score_go_hc))]
score_go_hc <- score_go_hc[!duplicated(names(score_go_hc))]

gsea_go_hc <- gseGO(
        geneList = score_go_hc, ont = "BP", OrgDb = org.Mm.eg.db, keyType = "SYMBOL",
        pvalueCutoff = 0.5, pAdjustMethod = "BH",
        by = "fgsea", scoreType = "pos",
        minGSSize = 50, maxGSSize = 500
)
cat("GO BP terms (UMI>=100):", nrow(gsea_go_hc@result), "\n")

# Compare top 20 terms between full and UMI-filtered
top_full <- head(gsea_go_v2@result$Description, 20)
top_hc <- head(gsea_go_hc@result$Description, 20)
cat("\nTop-20 overlap:", length(intersect(top_full, top_hc)), "/ 20\n")
cat("Terms in full but not in UMI-filtered:\n")
cat(paste(" -", setdiff(top_full, top_hc)), sep = "\n")
cat("\nTerms in UMI-filtered but not in full:\n")
cat(paste(" -", setdiff(top_hc, top_full)), sep = "\n")


## Method 1 vs Method 2 enrichment comparison


In [ ]:
# Load method 1 GSEA results for comparison
m1_dir <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult/ORA")
m1_metric <- "neglog10pval_optimized"

compare_gsea <- function(m2_res, m1_path, db_name, padj_cut = 0.05) {
    if (!file.exists(m1_path)) {
        cat(sprintf("  %s: Method 1 RDS not found\n", db_name))
        return(invisible(NULL))
    }
    m1_res <- readRDS(m1_path)
    m1_sig <- m1_res@result %>%
        filter(p.adjust < padj_cut) %>%
        pull(Description)
    m2_sig <- m2_res@result %>%
        filter(p.adjust < padj_cut) %>%
        pull(Description)
    cat(sprintf("\n%s (padj < %.2f):\n", db_name, padj_cut))
    cat(sprintf("  Method 1: %d sig terms\n", length(m1_sig)))
    cat(sprintf("  Method 2: %d sig terms\n", length(m2_sig)))
    cat(sprintf("  Overlap:  %d\n", length(intersect(m1_sig, m2_sig))))
    cat(sprintf("  M2 only:  %d\n", length(setdiff(m2_sig, m1_sig))))
    cat(sprintf("  M1 only:  %d\n", length(setdiff(m1_sig, m2_sig))))
    if (length(setdiff(m2_sig, m1_sig)) > 0) {
        cat("  New in M2:", paste(head(setdiff(m2_sig, m1_sig), 5), collapse = "; "), "\n")
    }
    if (length(setdiff(m1_sig, m2_sig)) > 0) {
        cat("  Lost from M1:", paste(head(setdiff(m1_sig, m2_sig), 5), collapse = "; "), "\n")
    }
}

cat("=== Method 1 vs Method 2 GSEA comparison ===\n")
compare_gsea(gsea_go_v2, file.path(m1_dir, paste0("gsea_go_", m1_metric, ".rds")), "GO BP")
compare_gsea(gsea_do_v2, file.path(m1_dir, paste0("gsea_do_", m1_metric, ".rds")), "DO")
compare_gsea(gsea_dgn_v2, file.path(m1_dir, paste0("gsea_dgn_", m1_metric, ".rds")), "DGN")
compare_gsea(gsea_kegg_v2, file.path(m1_dir, paste0("gsea_kegg_", m1_metric, ".rds")), "KEGG")
compare_gsea(gsea_reactome_v2, file.path(m1_dir, paste0("gsea_reactome_", m1_metric, ".rds")), "REACTOME")
compare_gsea(gsea_hallmark_v2, file.path(m1_dir, paste0("gsea_hallmark_", m1_metric, ".rds")), "Hallmark")
compare_gsea(gsea_hpo_v2, file.path(m1_dir, paste0("gsea_hpo_", m1_metric, ".rds")), "HPO")


## Save results


In [ ]:
saveRDS(gsea_go_v2, file.path(output_dir_v2, paste0("gsea_go_", metric_v2, ".rds")))
saveRDS(gsea_do_v2, file.path(output_dir_v2, paste0("gsea_do_", metric_v2, ".rds")))
saveRDS(gsea_dgn_v2, file.path(output_dir_v2, paste0("gsea_dgn_", metric_v2, ".rds")))
saveRDS(gsea_kegg_v2, file.path(output_dir_v2, paste0("gsea_kegg_", metric_v2, ".rds")))
saveRDS(gsea_reactome_v2, file.path(output_dir_v2, paste0("gsea_reactome_", metric_v2, ".rds")))
saveRDS(gsea_hallmark_v2, file.path(output_dir_v2, paste0("gsea_hallmark_", metric_v2, ".rds")))
saveRDS(gsea_hpo_v2, file.path(output_dir_v2, paste0("gsea_hpo_", metric_v2, ".rds")))

saveRDS(ora_go_v2, file.path(output_dir_v2, paste0("ora_go_", metric_v2, ".rds")))
saveRDS(ora_do_v2, file.path(output_dir_v2, paste0("ora_do_", metric_v2, ".rds")))
saveRDS(ora_kegg_v2, file.path(output_dir_v2, paste0("ora_kegg_", metric_v2, ".rds")))
saveRDS(ora_dgn_v2, file.path(output_dir_v2, paste0("ora_dgn_", metric_v2, ".rds")))
saveRDS(ora_reactome_v2, file.path(output_dir_v2, paste0("ora_reactome_", metric_v2, ".rds")))
saveRDS(ora_hallmark_v2, file.path(output_dir_v2, paste0("ora_hallmark_", metric_v2, ".rds")))
saveRDS(ora_hpo_v2, file.path(output_dir_v2, paste0("ora_hpo_", metric_v2, ".rds")))

saveRDS(gsea_go_hc, file.path(output_dir_v2, paste0("gsea_go_umi100_", metric_v2, ".rds")))
saveRDS(score_go_v2, file.path(output_dir_v2, paste0("score_go_", metric_v2, ".rds")))
saveRDS(score_human_v2, file.path(output_dir_v2, paste0("score_do_", metric_v2, ".rds")))
saveRDS(score_kegg_v2, file.path(output_dir_v2, paste0("score_kegg_", metric_v2, ".rds")))

cat("Saved all results to", output_dir_v2, "\n")


## Visualization (load-back section)

Run from here if results are already saved.


In [ ]:
# === Load-back cell: reload all saved results so viz section is self-contained ===
output_dir_v2 <- file.path(
        paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized"), "ORA"
)
metric_v2 <- "neglog10pval_chisq_optimized2"

gsea_go_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_go_", metric_v2, ".rds")))
gsea_do_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_do_", metric_v2, ".rds")))
gsea_dgn_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_dgn_", metric_v2, ".rds")))
gsea_kegg_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_kegg_", metric_v2, ".rds")))
gsea_reactome_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_reactome_", metric_v2, ".rds")))
gsea_hallmark_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_hallmark_", metric_v2, ".rds")))
gsea_hpo_v2 <- readRDS(file.path(output_dir_v2, paste0("gsea_hpo_", metric_v2, ".rds")))

ora_go_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_go_", metric_v2, ".rds")))
ora_do_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_do_", metric_v2, ".rds")))
ora_kegg_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_kegg_", metric_v2, ".rds")))
ora_dgn_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_dgn_", metric_v2, ".rds")))
ora_reactome_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_reactome_", metric_v2, ".rds")))
ora_hallmark_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_hallmark_", metric_v2, ".rds")))
ora_hpo_v2 <- readRDS(file.path(output_dir_v2, paste0("ora_hpo_", metric_v2, ".rds")))

score_go_v2 <- readRDS(file.path(output_dir_v2, paste0("score_go_", metric_v2, ".rds")))
score_human_v2 <- readRDS(file.path(output_dir_v2, paste0("score_do_", metric_v2, ".rds")))
score_kegg_v2 <- readRDS(file.path(output_dir_v2, paste0("score_kegg_", metric_v2, ".rds")))

mouse_to_human_genes <- readr::read_csv(paste0(DATA_ROOT, "/common_files/annotations/mouse_to_human_genes.txt"))
gsea_do_read_v2 <- setReadable(gsea_do_v2, "org.Hs.eg.db", "ENTREZID")
gsea_dgn_read_v2 <- setReadable(gsea_dgn_v2, "org.Hs.eg.db", "ENTREZID")
gsea_kegg_read_v2 <- setReadable(gsea_kegg_v2, "org.Mm.eg.db", "ENTREZID")
gsea_reactome_read_v2 <- setReadable(gsea_reactome_v2, "org.Hs.eg.db", "ENTREZID")

cat("All results loaded\n")


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10, repr.plot.res = 200)
options(warn = -1)

custom_theme <- theme_ipsum_rc() +
        theme(
                plot.title = element_text(size = 12, hjust = 0.5),
                panel.grid.major.x = element_blank(),
                panel.grid.minor.x = element_blank(),
                panel.grid.major.y = element_line(size = .05, color = "grey60")
        )


### GO BP


In [ ]:
p_dot <- dotplot(gsea_go_v2, showCategory = 20) + custom_theme +
        ggtitle("GO BP - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_go_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_go_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("GO BP - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_go_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_go_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_go_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_go_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_go_pw <- pairwise_termsim(gsea_go_v2)
p_emap <- emapplot(gsea_go_pw, showCategory = 30) +
        ggtitle("GO BP - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_go_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
p_cnet <- cnetplot(gsea_go_pw, showCategory = 5, foldChange = score_go_v2) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("GO BP - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_go_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_go_pw, showCategory = 30) +
        ggtitle("GO BP - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_go_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### KEGG


In [ ]:
p_dot <- dotplot(gsea_kegg_read_v2, showCategory = 20) + custom_theme +
        ggtitle("KEGG - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_kegg_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_kegg_read_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("KEGG - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_kegg_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_kegg_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_kegg_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_kegg_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_kegg_pw <- pairwise_termsim(gsea_kegg_read_v2)
p_emap <- emapplot(gsea_kegg_pw, showCategory = 30) +
        ggtitle("KEGG - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_kegg_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
fc_symbols <- score_kegg_v2
names(fc_symbols) <- gsea_kegg_read_v2@gene2Symbol[names(fc_symbols)]
gsea_kegg_read_pw <- pairwise_termsim(gsea_kegg_read_v2)
p_cnet <- cnetplot(gsea_kegg_read_pw, showCategory = 5, foldChange = fc_symbols) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("KEGG - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_kegg_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_kegg_pw, showCategory = 30) +
        ggtitle("KEGG - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_kegg_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### Disease Ontology


In [ ]:
p_dot <- dotplot(gsea_do_read_v2, showCategory = 20) + custom_theme +
        ggtitle("Disease Ontology - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_do_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_do_read_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("Disease Ontology - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_do_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_do_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_do_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_do_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_do_pw <- pairwise_termsim(gsea_do_read_v2)
p_emap <- emapplot(gsea_do_pw, showCategory = 30) +
        ggtitle("Disease Ontology - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_do_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
fc_symbols <- score_human_v2
names(fc_symbols) <- gsea_do_read_v2@gene2Symbol[names(fc_symbols)]
gsea_do_read_pw <- pairwise_termsim(gsea_do_read_v2)
p_cnet <- cnetplot(gsea_do_read_pw, showCategory = 5, foldChange = fc_symbols) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("Disease Ontology - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_do_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_do_pw, showCategory = 30) +
        ggtitle("Disease Ontology - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_do_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### DisGeNET


In [ ]:
p_dot <- dotplot(gsea_dgn_read_v2, showCategory = 20) + custom_theme +
        ggtitle("DisGeNET - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_dgn_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_dgn_read_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("DisGeNET - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_dgn_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_dgn_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_dgn_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_dgn_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_dgn_pw <- pairwise_termsim(gsea_dgn_read_v2)
p_emap <- emapplot(gsea_dgn_pw, showCategory = 30) +
        ggtitle("DisGeNET - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_dgn_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
fc_symbols <- score_human_v2
names(fc_symbols) <- gsea_dgn_read_v2@gene2Symbol[names(fc_symbols)]
gsea_dgn_read_pw <- pairwise_termsim(gsea_dgn_read_v2)
p_cnet <- cnetplot(gsea_dgn_read_pw, showCategory = 5, foldChange = fc_symbols) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("DisGeNET - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_dgn_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_dgn_pw, showCategory = 30) +
        ggtitle("DisGeNET - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_dgn_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### REACTOME


In [ ]:
p_dot <- dotplot(gsea_reactome_read_v2, showCategory = 20) + custom_theme +
        ggtitle("REACTOME - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_reactome_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_reactome_read_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("REACTOME - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_reactome_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_reactome_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_reactome_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_reactome_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_reactome_pw <- pairwise_termsim(gsea_reactome_read_v2)
p_emap <- emapplot(gsea_reactome_pw, showCategory = 30) +
        ggtitle("REACTOME - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_reactome_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
fc_symbols <- score_human_v2
names(fc_symbols) <- gsea_reactome_read_v2@gene2Symbol[names(fc_symbols)]
gsea_reactome_read_pw <- pairwise_termsim(gsea_reactome_read_v2)
p_cnet <- cnetplot(gsea_reactome_read_pw, showCategory = 5, foldChange = fc_symbols) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("REACTOME - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_reactome_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_reactome_pw, showCategory = 30) +
        ggtitle("REACTOME - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_reactome_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### MSigDB Hallmark


In [ ]:
p_dot <- dotplot(gsea_hallmark_v2, showCategory = 20) + custom_theme +
        ggtitle("MSigDB Hallmark - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_hallmark_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_hallmark_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("MSigDB Hallmark - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_hallmark_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_hallmark_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_hallmark_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_hallmark_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_hallmark_pw <- pairwise_termsim(gsea_hallmark_v2)
p_emap <- emapplot(gsea_hallmark_pw, showCategory = 30) +
        ggtitle("MSigDB Hallmark - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_hallmark_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
p_cnet <- cnetplot(gsea_hallmark_pw, showCategory = 5, foldChange = score_go_v2) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("MSigDB Hallmark - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_hallmark_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_hallmark_pw, showCategory = 30) +
        ggtitle("MSigDB Hallmark - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_hallmark_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### HPO


In [ ]:
p_dot <- dotplot(gsea_hpo_v2, showCategory = 20) + custom_theme +
        ggtitle("HPO - GSEA\nAPA changes over pseudotime (DirMult optimized_2)")
ggsave(file.path(output_dir_v2, paste0("dotplot_hpo_", metric_v2, ".png")),
        plot = p_dot, dpi = 400, width = 8, height = 10
)
p_dot


In [ ]:
p_ridge <- ridgeplot(gsea_hpo_v2, showCategory = 20, orderBy = "p.adjust", decreasing = TRUE) +
        custom_theme +
        ggtitle("HPO - GSEA\nAPA changes over pseudotime (DirMult optimized_2)") +
        xlab("-log10(p_value_chisq)") +
        geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4)
ggsave(file.path(output_dir_v2, paste0("ridgeplot_hpo_", metric_v2, ".png")),
        plot = p_ridge, dpi = 400, width = 9, height = 10
)
p_ridge


In [ ]:
n_top <- min(3, nrow(gsea_hpo_v2@result))
if (n_top > 0) {
        p_gsea <- gseaplot2(gsea_hpo_v2, geneSetID = 1:n_top, pvalue_table = TRUE)
        ggsave(file.path(output_dir_v2, paste0("gseaplot_hpo_top3_", metric_v2, ".png")),
                plot = p_gsea, dpi = 400, width = 10, height = 7
        )
        print(p_gsea)
}


In [ ]:
gsea_hpo_pw <- pairwise_termsim(gsea_hpo_v2)
p_emap <- emapplot(gsea_hpo_pw, showCategory = 30) +
        ggtitle("HPO - Enrichment map")
ggsave(file.path(output_dir_v2, paste0("emapplot_hpo_", metric_v2, ".png")),
        plot = p_emap, dpi = 400, width = 10, height = 10
)
p_emap


In [ ]:
p_cnet <- cnetplot(gsea_hpo_pw, showCategory = 5, foldChange = score_go_v2) +
        scale_color_viridis(name = "-log10(p)") +
        ggtitle("HPO - Gene-concept network (top 5 terms)")
ggsave(file.path(output_dir_v2, paste0("cnetplot_hpo_", metric_v2, ".png")),
        plot = p_cnet, dpi = 400, width = 12, height = 10
)
p_cnet


In [ ]:
p_tree <- treeplot(gsea_hpo_pw, showCategory = 30) +
        ggtitle("HPO - Hierarchical term clustering")
ggsave(file.path(output_dir_v2, paste0("treeplot_hpo_", metric_v2, ".png")),
        plot = p_tree, dpi = 400, width = 12, height = 10
)
p_tree


### ORA dotplots


In [ ]:
if (nrow(ora_go_v2@result) > 0 && sum(ora_go_v2@result$p.adjust < 0.05) > 0) {
        p_ora <- dotplot(ora_go_v2, showCategory = 20) + custom_theme +
                ggtitle("GO BP - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_go_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant GO BP ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_do_v2@result) > 0 && sum(ora_do_v2@result$p.adjust < 0.05) > 0) {
        ora_do_read_v2 <- setReadable(ora_do_v2, "org.Hs.eg.db", "ENTREZID")
        p_ora <- dotplot(ora_do_read_v2, showCategory = 20) + custom_theme +
                ggtitle("Disease Ontology - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_do_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant Disease Ontology ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_dgn_v2@result) > 0 && sum(ora_dgn_v2@result$p.adjust < 0.05) > 0) {
        ora_dgn_read_v2 <- setReadable(ora_dgn_v2, "org.Hs.eg.db", "ENTREZID")
        p_ora <- dotplot(ora_dgn_read_v2, showCategory = 20) + custom_theme +
                ggtitle("DisGeNET - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_dgn_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant DisGeNET ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_kegg_v2@result) > 0 && sum(ora_kegg_v2@result$p.adjust < 0.05) > 0) {
        p_ora <- dotplot(ora_kegg_v2, showCategory = 20) + custom_theme +
                ggtitle("KEGG - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_kegg_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant KEGG ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_reactome_v2@result) > 0 && sum(ora_reactome_v2@result$p.adjust < 0.05) > 0) {
        ora_reactome_read_v2 <- setReadable(ora_reactome_v2, "org.Hs.eg.db", "ENTREZID")
        p_ora <- dotplot(ora_reactome_read_v2, showCategory = 20) + custom_theme +
                ggtitle("REACTOME - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_reactome_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant REACTOME ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_hallmark_v2@result) > 0 && sum(ora_hallmark_v2@result$p.adjust < 0.05) > 0) {
        p_ora <- dotplot(ora_hallmark_v2, showCategory = 20) + custom_theme +
                ggtitle("MSigDB Hallmark - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_hallmark_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant MSigDB Hallmark ORA terms at padj < 0.05\n")
}


In [ ]:
if (nrow(ora_hpo_v2@result) > 0 && sum(ora_hpo_v2@result$p.adjust < 0.05) > 0) {
        p_ora <- dotplot(ora_hpo_v2, showCategory = 20) + custom_theme +
                ggtitle("HPO - ORA\n(padj_chisq < 0.05 genes)")
        ggsave(file.path(output_dir_v2, paste0("dotplot_ora_hpo_", metric_v2, ".png")),
                plot = p_ora, dpi = 400, width = 8, height = 8
        )
        p_ora
} else {
        cat("No significant HPO ORA terms at padj < 0.05\n")
}


### Leading-Edge Gene Overlap (UpSet)


In [ ]:
extract_le_genes <- function(gsea_res, padj_cutoff = 0.05) {
        sig <- gsea_res@result[gsea_res@result$p.adjust < padj_cutoff, ]
        if (nrow(sig) == 0) {
                return(character(0))
        }
        unique(unlist(strsplit(sig$core_enrichment, "/")))
}

le_go <- extract_le_genes(gsea_go_v2)
le_kegg <- extract_le_genes(gsea_kegg_read_v2)
le_do <- extract_le_genes(gsea_do_read_v2)
le_dgn <- extract_le_genes(gsea_dgn_read_v2)
le_reactome <- extract_le_genes(gsea_reactome_read_v2)
le_hallmark <- extract_le_genes(gsea_hallmark_v2)
le_hpo <- extract_le_genes(gsea_hpo_v2)

# Map human symbols to mouse symbols
human_to_mouse <- mouse_to_human_genes %>%
        filter(!is.na(`Human gene name`) & !is.na(`Gene name`)) %>%
        distinct(`Human gene name`, .keep_all = TRUE) %>%
        dplyr::select(`Human gene name`, `Gene name`)
h2m_map <- setNames(human_to_mouse$`Gene name`, human_to_mouse$`Human gene name`)

le_do_mouse <- unique(na.omit(h2m_map[le_do]))
le_dgn_mouse <- unique(na.omit(h2m_map[le_dgn]))
le_reactome_mouse <- unique(na.omit(h2m_map[le_reactome]))

le_list <- list(
        GO_BP = le_go, KEGG = le_kegg, DO = le_do_mouse,
        DGN = le_dgn_mouse, REACTOME = le_reactome_mouse,
        Hallmark = le_hallmark, HPO = le_hpo
)
le_list <- le_list[sapply(le_list, length) > 0]

cat("Leading-edge gene counts:\n")
for (nm in names(le_list)) cat(sprintf("  %s: %d\n", nm, length(le_list[[nm]])))

if (length(le_list) >= 2) {
        upset(fromList(le_list),
                order.by = "freq", nsets = length(le_list),
                text.scale = 1.2, point.size = 3, line.size = 0.8
        )
}
